# Étape 6.4-6.6 : Modélisation

Entraînement, comparaison des modèles, et génération de la soumission Kaggle.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.modeling import (
    compare_all_models, train_final_model, save_model,
    generate_submission, rmsle_original_scale, get_models,
    cross_validate_model, baseline_rmsle
)
from src.feature_engineering import FEATURE_COLS

train_feat = pd.read_csv('../data/processed/enriched_data.csv')
test_feat  = pd.read_csv('../data/processed/test_enriched.csv')

print(f"Train : {train_feat.shape}")
print(f"Test  : {test_feat.shape}")

In [ ]:
# Préparer X et y
X_train = train_feat[FEATURE_COLS].values
y_train = np.log1p(train_feat['prix'].values)  # log-space (target RMSLE)
X_test  = test_feat[FEATURE_COLS].values

print(f"X_train : {X_train.shape}")
print(f"y_train (log) : min={y_train.min():.2f}, max={y_train.max():.2f}, mean={y_train.mean():.2f}")
print(f"X_test  : {X_test.shape}")

# Vérification
assert not np.isnan(X_train).any(), "NaN dans X_train !"
assert not np.isnan(y_train).any(), "NaN dans y_train !"
assert not np.isnan(X_test).any(),  "NaN dans X_test !"
print("Aucun NaN dans les matrices")

## Baseline

In [ ]:
# Baseline : prédire la médiane pour tous
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
baseline_scores = []
for train_idx, val_idx in kf.split(X_train):
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    median_pred = np.full_like(y_val, np.median(y_tr))
    baseline_scores.append(rmsle_original_scale(y_val, median_pred))

baseline_score = np.mean(baseline_scores)
print(f"Baseline RMSLE (médiane) : {baseline_score:.4f}")

## Comparaison des modèles (KFold 5-fold)

In [ ]:
print("Entraînement de tous les modèles en cross-validation...")
results_df = compare_all_models(X_train, y_train, baseline_score)
print("\n=== RÉSULTATS ===")
print(results_df[['RMSLE', 'R2', 'RMSE_log']].round(4))

In [ ]:
# Graphique comparaison des modèles
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Filtrer la baseline pour les graphiques
non_baseline = results_df[results_df.index != 'Baseline (médiane)'].copy()
all_models = results_df.copy()

# RMSLE
ax = axes[0]
colors_bar = ['red' if idx == 'Baseline (médiane)' else 'steelblue' for idx in all_models.index]
bars = ax.bar(all_models.index, all_models['RMSLE'], color=colors_bar, edgecolor='white', alpha=0.85)
ax.axhline(baseline_score, color='red', linestyle='--', lw=1.5, label=f'Baseline: {baseline_score:.3f}')
ax.set_title('RMSLE (plus bas = meilleur)', fontsize=12)
ax.set_ylabel('RMSLE')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, all_models['RMSLE']):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.legend()

# R²
ax = axes[1]
r2_vals = non_baseline['R2'].fillna(0)
bars2 = ax.bar(non_baseline.index, r2_vals, color='teal', edgecolor='white', alpha=0.85)
ax.set_title('R² (plus haut = meilleur)', fontsize=12)
ax.set_ylabel('R²')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars2, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Étape 6.4 : Comparaison des Modèles', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/etape6_comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

## Meilleur modèle : entraînement final

In [ ]:
# Sélectionner le meilleur modèle (hors baseline)
non_baseline_results = results_df.drop('Baseline (médiane)', errors='ignore')
best_model_name = non_baseline_results['RMSLE'].idxmin()
print(f"Meilleur modèle : {best_model_name} (RMSLE = {non_baseline_results.loc[best_model_name, 'RMSLE']:.4f})")

# Ré-entraîner sur tout le train
models_dict = get_models()
best_model = models_dict[best_model_name]
best_model = train_final_model(best_model, X_train, y_train)
print(f"Modèle '{best_model_name}' entraîné sur {X_train.shape[0]} exemples.")

In [ ]:
# Feature importances (si disponible)
fig, ax = plt.subplots(figsize=(10, 8))

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)
    
    ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue', edgecolor='white')
    ax.set_title(f'Top 20 Feature Importances — {best_model_name}', fontsize=13)
    ax.set_xlabel('Importance')
elif hasattr(best_model, 'coef_'):
    coefs = np.abs(best_model.coef_)
    fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': coefs})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)
    
    ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue', edgecolor='white')
    ax.set_title(f'Top 20 Coefficients (valeur absolue) — {best_model_name}', fontsize=13)
    ax.set_xlabel('|Coefficient|')
else:
    ax.text(0.5, 0.5, 'Feature importances non disponibles', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)

plt.tight_layout()
plt.savefig('../outputs/figures/etape6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Réel vs prédit (sur le train complet)
y_pred_train = best_model.predict(X_train)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_train, y_pred_train, alpha=0.3, s=15, color='steelblue', label='Données')
min_val = min(y_train.min(), y_pred_train.min())
max_val = max(y_train.max(), y_pred_train.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r-', lw=2, label='y = x (parfait)')
ax.set_xlabel('log(1+prix) réel')
ax.set_ylabel('log(1+prix) prédit')
ax.set_title(f'Réel vs Prédit — {best_model_name}', fontsize=13)

train_rmsle = rmsle_original_scale(y_train, y_pred_train)
ax.text(0.05, 0.95, f'RMSLE train: {train_rmsle:.4f}', transform=ax.transAxes,
        verticalalignment='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/etape6_reel_vs_predit.png', dpi=150, bbox_inches='tight')
plt.show()

## Génération de la soumission

In [ ]:
submission = generate_submission(
    best_model, X_test,
    test_ids=test_feat['id'],
    output_path='../data/final/submission.csv'
)
print(submission.head(10))
print(f"\nNb lignes : {len(submission)}")
print(f"Prix min : {submission['prix'].min():,}")
print(f"Prix max : {submission['prix'].max():,}")
print(f"Prix médian : {submission['prix'].median():,}")

In [ ]:
# Sauvegarder le modèle
save_model(best_model, FEATURE_COLS, model_dir='../model/')
print("Pipeline complet terminé !")

In [ ]:
# Résumé final
print("=" * 60)
print("RÉSUMÉ DU PROJET CAPSTONE")
print("=" * 60)
print(f"Baseline RMSLE          : {baseline_score:.4f}")
print(f"Meilleur modèle         : {best_model_name}")
best_cv_rmsle = non_baseline_results.loc[best_model_name, 'RMSLE']
print(f"RMSLE CV (best model)   : {best_cv_rmsle:.4f}")
improvement = (baseline_score - best_cv_rmsle) / baseline_score * 100
print(f"Amélioration vs baseline: {improvement:.1f}%")
print(f"Fichier soumission       : data/final/submission.csv ({len(submission)} lignes)")
print(f"Modèle sauvegardé        : model/housing_model.pkl")
print("=" * 60)